# ETL Multi-Índice — Asesor Comercial Cibertec

Este notebook realiza el proceso completo de **Chunking → Embedding (Azure OpenAI 1536 dims) → Upsert en S3 Vectors (us-east-2)**
para los 4 índices de conocimiento del Asesor Comercial.

| Índice | Documentos |
|---|---|
| `idx-institucional` | RTB'S CIBERTEC + INDUCCIÓN FACULTADES |
| `idx-argumentario` | CIB ARGUMENTARIO 2026 V1.pdf |
| `idx-oferta-academica` | Comparativo Bach VS CT + TIPOS DE INGRESO |
| `idx-convenios` | CONVENIOS INSTITUCIONES |

## 0. Instalación de dependencias

In [ ]:
%pip install boto3 pypdf python-pptx langchain-text-splitters openai python-dotenv tqdm

## 1. Configuración — Actualiza los ARNs de tus índices aquí

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ─── AWS ───────────────────────────────────────────────────────────────────
AWS_REGION        = "us-east-2"
S3_BUCKET_NAME    = "poc-chat-prospect-cib"   # tu bucket de S3 Vectors

# ⚠️  ACTUALIZA estos nombres con los índices que creaste en la consola de AWS
INDEX_INSTITUCIONAL    = "idx-institucional"    # nombre del índice en S3 Vectors
INDEX_ARGUMENTARIO     = "idx-argumentario"
INDEX_OFERTA_ACADEMICA = "idx-oferta-academica"
INDEX_CONVENIOS        = "idx-convenios"

# ─── Azure OpenAI ──────────────────────────────────────────────────────────
AZURE_ENDPOINT    = os.environ.get("AZURE_ENDPOINT", "https://lau-prospectora-resource.cognitiveservices.azure.com/")
AZURE_API_KEY     = os.environ.get("AZURE_OPENAI_API_KEY", "")
AZURE_API_VERSION = "2024-12-01-preview"
EMBEDDING_MODEL   = "text-embedding-3-small-prospectora"  # deployment name
EMBEDDING_DIMS    = 1536

# ─── Chunking ──────────────────────────────────────────────────────────────
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100
BATCH_SIZE    = 500   # max vectores por llamada put_vectors

print("✅ Configuración cargada")
print(f"   Bucket: {S3_BUCKET_NAME} | Región: {AWS_REGION}")
print(f"   Embeddings: {EMBEDDING_MODEL} ({EMBEDDING_DIMS} dims)")

✅ Configuración cargada
   Bucket: poc-chat-prospect-cib | Región: us-east-2
   Embeddings: text-embedding-3-small-prospectora (1536 dims)


## 2. Clientes AWS y Azure OpenAI

In [5]:
import boto3
from openai import AzureOpenAI

# Cliente S3 Vectors
s3v = boto3.client(
    's3vectors',
    region_name=AWS_REGION,
    aws_access_key_id=os.environ.get("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.environ.get("AWS_SECRET_ACCESS_KEY")
)

# Cliente Azure OpenAI
az = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION
)

# Test rápido de embedding
test_emb = az.embeddings.create(input="test", model=EMBEDDING_MODEL).data[0].embedding
print(f"✅ Azure OpenAI OK — dimensiones: {len(test_emb)}")

✅ Azure OpenAI OK — dimensiones: 1536


## 3. Funciones de extracción, chunking y embedding

In [6]:
from pypdf import PdfReader
from pptx import Presentation
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import time

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

def extract_text_pdf(path: str) -> str:
    """Extrae texto de un PDF página por página."""
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

def extract_text_pptx(path: str) -> str:
    """Extrae texto de un PowerPoint slide por slide."""
    prs = Presentation(path)
    text = ""
    for slide in prs.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text.strip():
                text += shape.text.strip() + "\n"
    return text

def extract_text(path: str) -> str:
    """Detecta el tipo de archivo y extrae texto."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        return extract_text_pdf(path)
    elif ext in (".pptx", ".ppt"):
        return extract_text_pptx(path)
    else:
        raise ValueError(f"Formato no soportado: {ext}")

def get_chunks(paths: list[str]) -> list[str]:
    """Extrae y chunkea texto de múltiples archivos."""
    full_text = ""
    for path in paths:
        print(f"  📄 Extrayendo: {os.path.basename(path)}")
        full_text += extract_text(path) + "\n\n"
    chunks = splitter.split_text(full_text)
    print(f"  ✂️  {len(chunks)} chunks generados")
    return chunks

def get_embeddings(chunks: list[str]) -> list[list[float]]:
    """Genera embeddings con Azure OpenAI en lotes de 16."""
    embeddings = []
    batch_size = 16
    for i in tqdm(range(0, len(chunks), batch_size), desc="  🔢 Embeddings"):
        batch = chunks[i:i+batch_size]
        response = az.embeddings.create(input=batch, model=EMBEDDING_MODEL)
        embeddings.extend([d.embedding for d in response.data])
        time.sleep(0.2)  # evitar rate limit
    return embeddings

def upload_to_s3_vectors(index_name: str, chunks: list[str], embeddings: list[list[float]]):
    """Sube vectores al índice S3 Vectors en lotes."""
    vectors = [
        {
            "key": f"chunk-{i}",
            "data": {"float32": emb},
            "metadata": {"text": chunk}
        }
        for i, (chunk, emb) in enumerate(zip(chunks, embeddings))
    ]

    total = len(vectors)
    for start in tqdm(range(0, total, BATCH_SIZE), desc=f"  ⬆️  Subiendo a {index_name}"):
        batch = vectors[start:start+BATCH_SIZE]
        s3v.put_vectors(
            vectorBucketName=S3_BUCKET_NAME,
            indexName=index_name,
            vectors=batch
        )
    print(f"  ✅ {total} vectores subidos a '{index_name}'")

def process_index(index_name: str, file_paths: list[str]):
    """Pipeline completo para un índice: extrae → chunkea → embede → sube."""
    print(f"\n{'='*60}")
    print(f"🗂️  Procesando índice: {index_name}")
    print(f"{'='*60}")
    chunks = get_chunks(file_paths)
    embeddings = get_embeddings(chunks)
    upload_to_s3_vectors(index_name, chunks, embeddings)
    print(f"🎉 Índice '{index_name}' listo!\n")

print("✅ Funciones de ETL cargadas")

✅ Funciones de ETL cargadas


## 4. Definición de documentos por índice

In [8]:
DATA_DIR = "./data"

INDICES = {
    INDEX_INSTITUCIONAL: [
        f"{DATA_DIR}/RTB'S CIBERTEC (1).pdf",
        f"{DATA_DIR}/3.INDUCCIÓN DÍA 1 - RTB'S FACULTADES (1).pptx",
    ],
    INDEX_ARGUMENTARIO: [
        f"{DATA_DIR}/CIB ARGUMENTARIO 2026 V1.pdf",
    ],
    INDEX_OFERTA_ACADEMICA: [
        f"{DATA_DIR}/Comparativo Bach VS CT.pdf",
        f"{DATA_DIR}/TIPOS DE INGRESO POR MODALIDAD 2026.pdf",
    ],
    INDEX_CONVENIOS: [
        f"{DATA_DIR}/CONVENIOS INSTITUCIONES (1).pdf",
    ],
}

# Verificar que todos los archivos existen
all_ok = True
for index_name, paths in INDICES.items():
    for path in paths:
        exists = os.path.exists(path)
        status = "✅" if exists else "❌ NO ENCONTRADO"
        print(f"{status}  [{index_name}]  {os.path.basename(path)}")
        if not exists:
            all_ok = False

print()
print("✅ Todos los archivos encontrados. Listo para ETL." if all_ok else "❌ Corrige las rutas antes de continuar.")

✅  [idx-institucional]  RTB'S CIBERTEC (1).pdf
✅  [idx-institucional]  3.INDUCCIÓN DÍA 1 - RTB'S FACULTADES (1).pptx
✅  [idx-argumentario]  CIB ARGUMENTARIO 2026 V1.pdf
✅  [idx-oferta-academica]  Comparativo Bach VS CT.pdf
✅  [idx-oferta-academica]  TIPOS DE INGRESO POR MODALIDAD 2026.pdf
✅  [idx-convenios]  CONVENIOS INSTITUCIONES (1).pdf

✅ Todos los archivos encontrados. Listo para ETL.


## 5. Ejecutar ETL completo (los 4 índices)

> ⚠️ Esto puede tomar varios minutos dependiendo del tamaño de los documentos.
> El Argumentario tiene ~250 páginas, por lo que será el más largo.

In [10]:
for index_name, file_paths in INDICES.items():
    process_index(index_name, file_paths)

print("\n🏁 ETL completo. Los 4 índices están listos en S3 Vectors.")


🗂️  Procesando índice: idx-institucional
  📄 Extrayendo: RTB'S CIBERTEC (1).pdf
  📄 Extrayendo: 3.INDUCCIÓN DÍA 1 - RTB'S FACULTADES (1).pptx
  ✂️  23 chunks generados


  ⬆️  Subiendo a idx-institucional: 100%|██████████| 1/1 [00:01<00:00,  1.90s/it]


  ✅ 23 vectores subidos a 'idx-institucional'
🎉 Índice 'idx-institucional' listo!


🗂️  Procesando índice: idx-argumentario
  📄 Extrayendo: CIB ARGUMENTARIO 2026 V1.pdf
  ✂️  626 chunks generados


  ⬆️  Subiendo a idx-argumentario: 100%|██████████| 2/2 [00:05<00:00,  2.85s/it]


  ✅ 626 vectores subidos a 'idx-argumentario'
🎉 Índice 'idx-argumentario' listo!


🗂️  Procesando índice: idx-oferta-academica
  📄 Extrayendo: Comparativo Bach VS CT.pdf
  📄 Extrayendo: TIPOS DE INGRESO POR MODALIDAD 2026.pdf
  ✂️  33 chunks generados


  ⬆️  Subiendo a idx-oferta-academica: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]


  ✅ 33 vectores subidos a 'idx-oferta-academica'
🎉 Índice 'idx-oferta-academica' listo!


🗂️  Procesando índice: idx-convenios
  📄 Extrayendo: CONVENIOS INSTITUCIONES (1).pdf
  ✂️  8 chunks generados


  ⬆️  Subiendo a idx-convenios: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]

  ✅ 8 vectores subidos a 'idx-convenios'
🎉 Índice 'idx-convenios' listo!


🏁 ETL completo. Los 4 índices están listos en S3 Vectors.


## 5b. Ejecutar un índice individual (opcional)

Si solo quieres re-indexar un índice específico, ejecuta solo la celda correspondiente.

In [ ]:
# Ejecutar solo idx-institucional
process_index(INDEX_INSTITUCIONAL, INDICES[INDEX_INSTITUCIONAL])

In [ ]:
# Ejecutar solo idx-argumentario (el más grande, ~250 páginas)
process_index(INDEX_ARGUMENTARIO, INDICES[INDEX_ARGUMENTARIO])

In [ ]:
# Ejecutar solo idx-oferta-academica
process_index(INDEX_OFERTA_ACADEMICA, INDICES[INDEX_OFERTA_ACADEMICA])

In [ ]:
# Ejecutar solo idx-convenios
process_index(INDEX_CONVENIOS, INDICES[INDEX_CONVENIOS])

## 6. Verificar vectores subidos por índice

In [11]:
print("📊 Resumen de vectores en S3 Vectors:\n")

for index_name in INDICES.keys():
    try:
        response = s3v.list_vectors(
            vectorBucketName=S3_BUCKET_NAME,
            
            indexName=index_name,
        )
        count = len(response.get('vectors', []))
        print(f"  🗂️  {index_name:<30} → {count:>4} vectores")
    except Exception as e:
        print(f"  ❌  {index_name:<30} → Error: {e}")

📊 Resumen de vectores en S3 Vectors:

  🗂️  idx-institucional              →   23 vectores
  🗂️  idx-argumentario               →  500 vectores
  🗂️  idx-oferta-academica           →   33 vectores
  🗂️  idx-convenios                  →    8 vectores


## 7. Test de búsqueda semántica en un índice

In [12]:
def search(index_name: str, query: str, top_k: int = 3):
    """Hace una búsqueda semántica en el índice indicado."""
    query_emb = az.embeddings.create(input=query, model=EMBEDDING_MODEL).data[0].embedding
    response = s3v.query_vectors(
        vectorBucketName=S3_BUCKET_NAME,
        indexName=index_name,
        queryVector={"float32": query_emb},
        topK=top_k,
        returnMetadata=True
    )
    results = response.get('vectors', [])
    print(f"\n🔍 Query: '{query}'  →  {index_name}")
    print(f"   {len(results)} resultados:\n")
    for i, r in enumerate(results, 1):
        score = round(1 - r.get('distance', 1), 4)
        text = r.get('metadata', {}).get('text', '')[:300]
        print(f"   [{i}] Score: {score}")
        print(f"       {text}...\n")

# Prueba por índice
search(INDEX_INSTITUCIONAL,    "¿Cuáles son las ventajas de estudiar en Cibertec?")
search(INDEX_ARGUMENTARIO,     "¿Cómo manejar la objeción de precio alto?")
search(INDEX_OFERTA_ACADEMICA, "¿Cuál es la diferencia entre técnica y bachiller?")
search(INDEX_CONVENIOS,        "¿Qué convenios tiene Cibertec con empresas?")


🔍 Query: '¿Cuáles son las ventajas de estudiar en Cibertec?'  →  idx-institucional
   2 resultados:

   [1] Score: 0
       VENTAJAS DIFERENCIALES
EMPLEABILIDAD
- 9 de cada 10 egresados de Cibertec consiguen trabajo en su primer año de egreso. 
- Obtén certificaciones modulares y de empleabilidad que fortalecen tu perfil profesional y responden a las 
demandas del mercado laboral.
FLEXIBILIDAD
Estudia a tu ritmo con clas...

   [2] Score: 0
       CONTINUIDAD DE ESTUDIOS 
EGRESADOS CIBERTEC
En CIBERTEC ayudamos a nuestros egresados a seguir desarrollándose como profesionales a 
través de los convenios de continuidad con universidades nacionales y extranjeras.
• Dirigidos a titulados o en proceso de titulación.
• Las universidades reconocen lo...


🔍 Query: '¿Cómo manejar la objeción de precio alto?'  →  idx-argumentario
   3 resultados:

   [1] Score: 0
       económica, asesorar clientes y optimizar procesos financieros, desempeñándose con solvencia en bancos y 
entidades financieras